# A1: SSM Scale Ablation (VMamba-Tiny / Small / Base)

**Author:** Mridankan Mandal

**Competition:** [CSIRO Image2Biomass](https://www.kaggle.com/competitions/csiro-biomass)

---

## Overview

This ablation evaluates the impact of SSM model scale on biomass estimation. Three VMamba variants are compared: Tiny (~27M), Small (~50M), and Base (~89M). All other components remain identical.

### Training Protocol
- **Seed:** 17 (deterministic)
- **Epochs:** 50 (max)
- **Early Stopping Patience:** 10 epochs

In [ ]:
# Install dependencies and download VMamba weights (Tiny, Small, Base)
!pip install -q huggingface_hub safetensors timm
!pip install -q git+https://github.com/MzeroMiko/VMamba.git || echo 'VMamba install failed, will use fallback'

# Download VMamba weights from HuggingFace
from huggingface_hub import hf_hub_download
import os

vmamba_ckpt_dir = "/kaggle/working/vmamba_checkpoints"
os.makedirs(vmamba_ckpt_dir, exist_ok=True)

vmamba_weights = {}
variants = {
    'tiny': 'vssmtiny_dp01_ckpt_epoch_292.pth',
    'small': 'vssmsmall_dp03_ckpt_epoch_238.pth',
    'base': 'vssm_base_0229_ckpt_epoch_237.pth'
}

for variant_name, filename in variants.items():
    try:
        weight_path = hf_hub_download(
            repo_id="MzeroMiko/VMamba",
            filename=filename,
            cache_dir=vmamba_ckpt_dir
        )
        vmamba_weights[variant_name] = weight_path
        print(f"VMamba-{variant_name.capitalize()} weights downloaded to: {weight_path}")
    except Exception as e:
        print(f"Failed to download VMamba-{variant_name}: {e}")
        vmamba_weights[variant_name] = ""

# VMamba imports
try:
    from vmamba import VSSM as VMambaModel
    VMAMBA_AVAILABLE = True
    print('[OK] VMamba package loaded successfully')
except ImportError:
    VMAMBA_AVAILABLE = False
    print('[WARN] vmamba package not found. Using timm ViT fallback.')

import os
import gc
import math
import random
import warnings
import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from tqdm.auto import tqdm
from typing import Optional

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image
import timm

warnings.filterwarnings('ignore')

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision("high")

print(" Imports complete")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"timm version: {timm.__version__}")

# --- VMamba configs (official MzeroMiko/VMamba v2, ssm_ratio=2.0) ---
VMAMBA_CONFIGS = {
    'vmamba_tiny': {
        'depths': [2, 2, 5, 2],
        'dims': 96,
        'num_features': 768,
        'weight_key': 'tiny',
    },
    'vmamba_small': {
        'depths': [2, 2, 15, 2],
        'dims': 96,
        'num_features': 768,
        'weight_key': 'small',
    },
    'vmamba_base': {
        'depths': [2, 2, 15, 2],
        'dims': 128,
        'num_features': 1024,
        'weight_key': 'base',
    },
}

class CFG:
    BASE_PATH = '/kaggle/input/competitions/csiro-biomass'
    TRAIN_CSV = os.path.join(BASE_PATH, 'train.csv')
    TRAIN_IMAGE_DIR = os.path.join(BASE_PATH, 'train')
    TEST_CSV = os.path.join(BASE_PATH, 'test.csv')
    TEST_IMAGE_DIR = os.path.join(BASE_PATH, 'test')
    
    MODEL_DIR = ''
    OUTPUT_DIR = ''
    
    # --- A1 Ablation Configuration ---
    MODEL_NAME = 'vmamba_base'    # Updated per-variant in the sweep loop
    VMAMBA_VARIANT = 'vmamba_base'
    
    SEED=17
    N_FOLDS = 5
    FOLDS_TO_TRAIN = [0, 1, 2, 3, 4]
    
    IMG_SIZE = 512
    BATCH_SIZE = 6
    NUM_WORKERS = 2
    
    EPOCHS=50
    WARMUP_EPOCHS=5
    LR_BACKBONE = 1e-5
    LR_HEAD = 5e-4
    WD = 1e-2
    
    CLIP_GRAD_NORM = 1.0
    DROPOUT = 0.2
    
    EARLY_STOPPING_PATIENCE=10
    
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    TARGET_COLS = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(CFG.MODEL_DIR, exist_ok=True)

def seed_everything(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

seed_everything()

print(f"\n{'='*60}")
print("CONFIGURATION - A1 SSM SCALE ABLATION")
print(f"{'='*60}")
print(f"Device: {CFG.DEVICE}")
print(f"Starting variant: {CFG.MODEL_NAME}")
print(f"Image Size: {CFG.IMG_SIZE}")
print(f"Batch Size: {CFG.BATCH_SIZE}")
print(f"Epochs: {CFG.EPOCHS}")
print(f"Folds: {CFG.N_FOLDS}")

print(f"\n{'='*60}")
print("STEP 1: Loading Data")
print(f"{'='*60}")

def load_train_data():
    df = pd.read_csv(CFG.TRAIN_CSV)
    df['image_id'] = df['sample_id'].str.split('__').str[0]
    
    df_wide = df.pivot_table(
        index=['image_id', 'image_path'],
        columns='target_name',
        values='target',
        aggfunc='first'
    ).reset_index()
    
    for col in CFG.TARGET_COLS:
        if col not in df_wide.columns:
            df_wide[col] = 0.0
    
    df_wide['total_bin'] = pd.qcut(df_wide['Dry_Total_g'], q=5, labels=False, duplicates='drop')
    
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    df_wide['fold'] = -1
    
    for fold, (_, val_idx) in enumerate(sgkf.split(df_wide, df_wide['total_bin'], groups=df_wide['image_id'])):
        df_wide.loc[val_idx, 'fold'] = fold
    
    print(f" Loaded {len(df_wide)} training images")
    print(f"Fold distribution:\n{df_wide['fold'].value_counts().sort_index()}")
    return df_wide

def load_test_data():
    df = pd.read_csv(CFG.TEST_CSV)
    df['image_id'] = df['sample_id'].str.split('__').str[0]
    df_unique = df.drop_duplicates('image_id')[['image_id', 'image_path']].reset_index(drop=True)
    print(f" Loaded {len(df_unique)} test images")
    return df_unique

train_df = load_train_data()
test_df = load_test_data()

def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])


# ── Image caching: pre-split + pre-resize to /tmp for fast loading ──
def prepare_image_cache(img_dir, img_size, df):
    """Pre-split and pre-resize all images to /tmp for fast epoch loading."""
    import json as _json
    cache_dir = f"/tmp/biomass_cache_{img_size}"
    manifest_path = os.path.join(cache_dir, "_manifest.json")
    unique_images = df['image_path'].unique()
    needed = len(unique_images)
    if os.path.exists(manifest_path):
        try:
            with open(manifest_path) as f:
                manifest = _json.load(f)
            if manifest.get('count') == needed and manifest.get('img_size') == img_size:
                print(f"[Cache] Ready: {cache_dir} ({needed} images, {img_size}px)")
                return cache_dir
        except Exception:
            pass
    os.makedirs(cache_dir, exist_ok=True)
    print(f"[Cache] Building image cache ({needed} images -> {img_size}x{img_size}) ...")
    cached = 0
    for img_rel in tqdm(unique_images, desc='Caching images'):
        img_name = os.path.basename(img_rel)
        base = os.path.splitext(img_name)[0]
        left_file = os.path.join(cache_dir, f"{base}_L.npy")
        right_file = os.path.join(cache_dir, f"{base}_R.npy")
        if os.path.exists(left_file) and os.path.exists(right_file):
            cached += 1
            continue
        full_path = os.path.join(img_dir, img_name)
        img = cv2.imread(full_path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        mid = w // 2
        left = cv2.resize(img[:, :mid], (img_size, img_size), interpolation=cv2.INTER_AREA)
        right = cv2.resize(img[:, mid:], (img_size, img_size), interpolation=cv2.INTER_AREA)
        np.save(left_file, left)
        np.save(right_file, right)
    with open(manifest_path, 'w') as f:
        _json.dump({'count': needed, 'img_size': img_size}, f)
    print(f"[Cache] Done: {cached} reused, {needed - cached} newly cached")
    return cache_dir

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, cache_dir=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
        self.paths = df['image_path'].values
        self.labels = df[CFG.TARGET_COLS].values.astype(np.float32)
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = os.path.basename(self.paths[idx])
        base = os.path.splitext(img_name)[0]
        
        if self.cache_dir:
            left = np.load(os.path.join(self.cache_dir, f"{base}_L.npy"))
            right = np.load(os.path.join(self.cache_dir, f"{base}_R.npy"))
        else:
            path = os.path.join(self.img_dir, img_name)
            img = cv2.imread(path)
            if img is None:
                img = np.zeros((1000, 2000, 3), dtype=np.uint8)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            h, w, _ = img.shape
            mid = w // 2
            left = img[:, :mid]
            right = img[:, mid:]
        
        if self.transform:
            left = self.transform(image=left)['image']
            right = self.transform(image=right)['image']
        
        label = torch.from_numpy(self.labels[idx])
        return left, right, label

print(" Dataset class defined (NO image cleaning)")

# ── VMamba Backbone (real VSSM) ──
class VMambaBackbone(nn.Module):
    """VMamba backbone using vmamba.VSSM with scale-dependent configs."""
    def __init__(self, variant='vmamba_base', pretrained_path=''):
        super().__init__()
        cfg = VMAMBA_CONFIGS[variant]
        
        if VMAMBA_AVAILABLE:
            self.model = VMambaModel(
                depths=cfg['depths'],
                dims=cfg['dims'],
                ssm_d_state=1,
                ssm_ratio=2.0,
                ssm_dt_rank="auto",
                ssm_act_layer="silu",
                ssm_conv=3,
                ssm_conv_bias=False,
                ssm_init="v0",
                forward_type="v05_noz",
                mlp_ratio=4.0,
                mlp_act_layer="gelu",
                patch_norm=True,
                norm_layer="ln2d",
                downsample_version="v3",
                patchembed_version="v2",
            )
            self.num_features = cfg['num_features']
            
            # Load pretrained weights
            if not pretrained_path:
                pretrained_path = vmamba_weights.get(cfg['weight_key'], '')
            if pretrained_path and os.path.exists(pretrained_path):
                ckpt = torch.load(pretrained_path, map_location='cpu', weights_only=False)
                if 'model' in ckpt:
                    ckpt = ckpt['model']
                ckpt = {k: v for k, v in ckpt.items() if not k.startswith('classifier.')}
                self.model.load_state_dict(ckpt, strict=False)
                print(f"Loaded VMamba-{variant} weights from {pretrained_path}")
            self._use_vssm = True
        else:
            print(f"[WARN] vmamba not available — using timm ViT fallback for {variant}")
            self.model = timm.create_model(
                'vit_base_patch16_224.augreg_in21k_ft_in1k',
                pretrained=True, num_classes=0, global_pool='',
                img_size=512,
            )
            self.num_features = self.model.num_features
            self._use_vssm = False
        print(f"VMamba backbone ({variant}), features={self.num_features}")

    def forward(self, x):
        if self._use_vssm:
            x = self.model.patch_embed(x)
            if self.model.pos_embed is not None:
                x = x + self.model.pos_embed
            for layer in self.model.layers:
                x = torch.utils.checkpoint.checkpoint(layer, x, use_reentrant=False)
            x = self.model.classifier.norm(x)
            if x.dim() == 4:
                B, C, H, W = x.shape
                x = x.permute(0, 2, 3, 1).reshape(B, H * W, C)
            return x
        return self.model(x)

# ── Mamba SSM Fusion (fallback to GatedDepthwiseConv if mamba_ssm unavailable) ──
try:
    from mamba_ssm import Mamba
    MAMBA_SSM_AVAILABLE = True
    print('[OK] mamba_ssm loaded')
except ImportError:
    MAMBA_SSM_AVAILABLE = False
    print('[INFO] mamba_ssm not available, using GatedDepthwiseConvBlock fusion')

class GatedDepthwiseConvBlock(nn.Module):
    """Gated depthwise-conv fusion block (fallback)."""
    def __init__(self, dim, kernel_size=5, dropout=0.1, **kwargs):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size // 2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        g = torch.sigmoid(self.gate(x))
        x = x * g
        x = x.transpose(1, 2)
        x = self.dwconv(x)
        x = x.transpose(1, 2)
        x = self.proj(x)
        x = self.drop(x)
        return shortcut + x

class MambaFusionBlock(nn.Module):
    """Real Mamba SSM fusion block (hardware-accelerated)."""
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, dropout=0.1, **kwargs):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        x = self.mamba(x)
        x = self.drop(x)
        return shortcut + x

def _make_head(nf, dropout):
    return nn.Sequential(
        nn.Linear(nf, nf // 2), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(nf // 2, 1), nn.Softplus()
    )

class BiomassModelVMamba(nn.Module):
    """Dual-view biomass model with VMamba backbone + fusion."""
    def __init__(self, variant='vmamba_base', pretrained_path='', pretrained=True):
        super().__init__()
        self.backbone = VMambaBackbone(variant=variant, pretrained_path=pretrained_path)
        nf = self.backbone.num_features
        
        FusionBlock = MambaFusionBlock if MAMBA_SSM_AVAILABLE else GatedDepthwiseConvBlock
        fusion_label = 'MambaSSM' if MAMBA_SSM_AVAILABLE else 'GatedDepthwiseConv'
        print(f"Fusion: 2x {fusion_label}Block(dim={nf})")
        
        self.fusion = nn.Sequential(
            FusionBlock(nf, dropout=CFG.DROPOUT),
            FusionBlock(nf, dropout=CFG.DROPOUT),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = _make_head(nf, CFG.DROPOUT)
        self.head_dead = _make_head(nf, CFG.DROPOUT)
        self.head_clover = _make_head(nf, CFG.DROPOUT)

    def forward(self, left, right):
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x_cat = torch.cat([x_l, x_r], dim=1)
        x_fused = self.fusion(x_cat)
        x_pool = self.pool(x_fused.transpose(1, 2)).flatten(1)
        
        green = self.head_green(x_pool)
        dead = self.head_dead(x_pool)
        clover = self.head_clover(x_pool)
        gdm = green + clover
        total = gdm + dead
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)

print(" VMamba model architecture defined")

def biomass_loss(preds, labels):
    huber = nn.SmoothL1Loss(beta=5.0)
    return huber(preds, labels)

def weighted_r2_score(y_true, y_pred):
    weights = np.array([0.1, 0.1, 0.1, 0.2, 0.5])
    r2_scores = []
    for i in range(y_true.shape[1]):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        r2_scores.append(r2)
    r2_scores = np.array(r2_scores)
    weighted = np.sum(r2_scores * weights) / np.sum(weights)
    return weighted, r2_scores

def build_optimizer(model):
    backbone_params = list(model.backbone.parameters())
    backbone_ids = {id(p) for p in backbone_params}
    head_params = [p for p in model.parameters() if id(p) not in backbone_ids]
    return optim.AdamW([
        {'params': backbone_params, 'lr': CFG.LR_BACKBONE},
        {'params': head_params, 'lr': CFG.LR_HEAD}
    ], weight_decay=CFG.WD, fused=True)

def build_scheduler(optimizer, total_steps):
    def lr_lambda(step):
        warmup_steps = CFG.WARMUP_EPOCHS * (total_steps // CFG.EPOCHS)
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)

scaler = GradScaler('cuda')

def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc='Training')
    for i, (left, right, labels) in enumerate(pbar):
        left = left.to(device, non_blocking=True, memory_format=torch.channels_last)
        right = right.to(device, non_blocking=True, memory_format=torch.channels_last)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast('cuda'):
            preds = model(left, right)
            loss = biomass_loss(preds, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{total_loss/(i+1):.4f}'})
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    for left, right, labels in tqdm(loader, desc='Validating'):
        left = left.to(device, non_blocking=True, memory_format=torch.channels_last)
        right = right.to(device, non_blocking=True, memory_format=torch.channels_last)
        with autocast('cuda'):
            preds = model(left, right)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    return weighted_r2_score(all_labels, all_preds)

print(" Training functions defined")

print(f"\n{'='*60}")
print("STEP 2: Training VMamba Models (Scale Sweep)")
print(f"{'='*60}")

def train_fold(fold, train_df, variant):
    print(f"\n{'='*60}")
    print(f"TRAINING FOLD {fold} — {variant}")
    print(f"{'='*60}")
    
    train_data = train_df[train_df['fold'] != fold].reset_index(drop=True)
    val_data = train_df[train_df['fold'] == fold].reset_index(drop=True)
    print(f"Train: {len(train_data)}, Val: {len(val_data)}")
    
    cache_dir = prepare_image_cache(CFG.TRAIN_IMAGE_DIR, CFG.IMG_SIZE, train_df)
    train_dataset = BiomassDataset(train_data, CFG.TRAIN_IMAGE_DIR, get_train_transforms(), cache_dir=cache_dir)
    val_dataset = BiomassDataset(val_data, CFG.TRAIN_IMAGE_DIR, get_val_transforms(), cache_dir=cache_dir)
    
    train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True,
                              num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True,
                              persistent_workers=CFG.NUM_WORKERS > 0,
                              prefetch_factor=2 if CFG.NUM_WORKERS > 0 else None)
    val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False,
                            num_workers=CFG.NUM_WORKERS, pin_memory=True,
                            persistent_workers=CFG.NUM_WORKERS > 0,
                            prefetch_factor=2 if CFG.NUM_WORKERS > 0 else None)
    
    model = BiomassModelVMamba(variant=variant).to(CFG.DEVICE)
    model = model.to(memory_format=torch.channels_last)
    
    optimizer = build_optimizer(model)
    total_steps = len(train_loader) * CFG.EPOCHS
    scheduler = build_scheduler(optimizer, total_steps)
    
    best_r2 = -float('inf')
    best_epoch = 0
    epochs_without_improvement = 0
    start_epoch = 0
    
    # --- Resume from checkpoint if available ---
    ckpt_path = os.path.join(CFG.MODEL_DIR, f"fold{fold}_checkpoint.pth")
    if os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        start_epoch = ckpt['epoch'] + 1
        best_r2 = ckpt['best_r2']
        best_epoch = ckpt['best_epoch']
        epochs_without_improvement = ckpt['patience_ctr']
        for _ in range(start_epoch * len(train_loader)):
            scheduler.step()
        print(f"[Resume] Loaded checkpoint: epoch {start_epoch}, best R²={best_r2:.4f}")
    
    epoch_pbar = tqdm(range(start_epoch, CFG.EPOCHS), desc=f'Fold {fold} Epochs')
    for epoch in epoch_pbar:
        print(f"\nEpoch {epoch + 1}/{CFG.EPOCHS}")
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, CFG.DEVICE)
        val_r2, per_r2 = validate(model, val_loader, CFG.DEVICE)
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val R²: {val_r2:.4f}")
        print(f"Per-target: Green={per_r2[0]:.3f}, Dead={per_r2[1]:.3f}, Clover={per_r2[2]:.3f}, GDM={per_r2[3]:.3f}, Total={per_r2[4]:.3f}")
        epoch_pbar.set_postfix({'loss': f'{train_loss:.4f}', 'val_r2': f'{val_r2:.4f}', 'best_r2': f'{best_r2:.4f}'})
        if val_r2 > best_r2:
            best_r2 = val_r2
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            save_path = f"{CFG.MODEL_DIR}/fold{fold}_best.pth"
            torch.save(model.state_dict(), save_path)
            print(f" Saved best model (R²={best_r2:.4f})")
        else:
            epochs_without_improvement += 1
            print(f"No improvement for {epochs_without_improvement} epoch(s)")
            if epochs_without_improvement >= CFG.EARLY_STOPPING_PATIENCE:
                print(f"Early stopping triggered after {epoch + 1} epochs")
                break
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'best_r2': best_r2,
            'best_epoch': best_epoch,
            'patience_ctr': epochs_without_improvement,
        }, os.path.join(CFG.MODEL_DIR, f"fold{fold}_checkpoint.pth"))
    
    # Cleanup and mark done
    ckpt_cleanup = os.path.join(CFG.MODEL_DIR, f"fold{fold}_checkpoint.pth")
    if os.path.exists(ckpt_cleanup):
        os.remove(ckpt_cleanup)
    import json as _json
    with open(os.path.join(CFG.MODEL_DIR, f"fold{fold}_done.json"), 'w') as f:
        _json.dump({'fold': fold, 'best_r2': float(best_r2), 'best_epoch': best_epoch}, f)
    print(f"\nFold {fold} Best: R²={best_r2:.4f} at epoch {best_epoch}")
    del model, optimizer, scheduler, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return best_r2

# --- Sweep all three VMamba scales ---
all_results = {}
for variant in ['vmamba_tiny', 'vmamba_small', 'vmamba_base']:
    print(f"\n{'#'*60}")
    print(f"  A1 ABLATION: {variant.upper()}")
    print(f"{'#'*60}")
    
    CFG.MODEL_NAME = variant
    CFG.VMAMBA_VARIANT = variant
    CFG.MODEL_DIR = f'/kaggle/working/A1_{variant}'
    CFG.OUTPUT_DIR = CFG.MODEL_DIR
    os.makedirs(CFG.MODEL_DIR, exist_ok=True)
    
    fold_scores = []
    for fold in CFG.FOLDS_TO_TRAIN:
        done_path = os.path.join(CFG.MODEL_DIR, f"fold{fold}_done.json")
        if os.path.exists(done_path):
            import json as _json
            with open(done_path) as f:
                done_info = _json.load(f)
            score = done_info['best_r2']
            print(f"\n[Resume] Fold {fold} already complete (R²={score:.4f}), skipping")
        else:
            score = train_fold(fold, train_df, variant)
        fold_scores.append(score)
    
    mean_r2 = np.mean(fold_scores)
    std_r2 = np.std(fold_scores)
    all_results[variant] = {'fold_scores': fold_scores, 'mean': mean_r2, 'std': std_r2}
    print(f"\n{variant}: Mean CV R² = {mean_r2:.4f} ± {std_r2:.4f}")
    
    import json
    with open(f'{CFG.OUTPUT_DIR}/training_summary.json', 'w') as f:
        json.dump({'model': variant, 'fold_scores': fold_scores, 'mean_cv': float(mean_r2), 'std_cv': float(std_r2)}, f, indent=2)

print(f"\n{'='*60}")
print("A1 SCALE SWEEP COMPLETE")
print(f"{'='*60}")
for v, r in all_results.items():
    print(f"  {v}: {r['mean']:.4f} ± {r['std']:.4f}")
print(f"\nModels saved to: /kaggle/working/A1_*/")